**1. Goal of this notebook**

By the end, you should understand this transformation:

``` text

Real image
   ↓
Width × Height
   ↓
Pixels
   ↓
RGB channels
   ↓
NumPy array
   ↓
PyTorch tensor

```

We are not training a model yet.

First, we need to understand exactly what the model will receive as input.

##### 2. Images have dimensions

Suppose we have a product image:

- Width  = 640 pixels
- Height = 480 pixels

That means the image contains: 640 × 480 = 307,200 pixels

For an RGB image, each pixel has three channel values:

- Red
- Green
- Blue

So the image contains: 640 × 480 × 3 = 921,600 numerical values

Already we can see why images contain much more numerical information than they appear to.

##### 3. Loading an image with Python

We'll commonly use Pillow (PIL) for basic image operations.

from PIL import Image

image_path = "/path/to/sample_image.jpg"
image = Image.open(image_path)
print(image)

You might see something similar to:

<PIL.JpegImagePlugin.JpegImageFile
 image mode=RGB
 size=640x480>

Two important pieces of information are:

- mode = RGB
- size = 640 × 480

We can inspect them directly:

print("Image size:", image.size)
print("Image mode:", image.mode)

Output:

- Image size: (640, 480)
- Image mode: RGB

Notice that PIL reports: (width, height)

not height first.

##### 4. What does RGB mode mean?

If image.mode returns: RGB

each pixel has three values: (R, G, B)

For example: (120, 85, 60)

means:

- Red   = 120
- Green = 85
- Blue  = 60

If we retrieve one pixel:

pixel = image.getpixel((100, 50))

print(pixel)

we might get: (120, 85, 60)

The coordinates: (100, 50)

mean approximately:

- x = 100 → horizontal position
- y = 50  → vertical position

So we're asking:

What RGB values are stored at this particular location?

##### 5. Image → NumPy array

Now we make an important transition.

``` text

import numpy as np

image_array = np.array(image)

print(image_array.shape)

```

For our example: (480, 640, 3)

What do those numbers mean?

- 480 = height
- 640 = width
- 3   = RGB channels

Therefore:

NumPy image shape

[height, width, channels]

[480, 640, 3]

This is our first important shape convention.

##### 6. What does the NumPy array actually contain?

Let's inspect it: print(image_array)

Conceptually, we're seeing something like:

``` text

[
   [
      [120, 85, 60],
      [122, 87, 61],
      [125, 90, 65],
      ...
   ],

   [
      [118, 83, 58],
      [121, 86, 60],
      ...
   ],

   ...
]

```
Each: [120, 85, 60] is one pixel.

And each number is a channel intensity.

So:

``` text

Image
 ↓
NumPy array
 ↓
Numbers

```
The computer doesn't receive: "Metal plate with scratch."

It receives numerical values arranged spatially.

##### 7. Understanding the three dimensions

Let's use a tiny image instead.

Suppose:

- Height = 2
- Width  = 3
- RGB    = 3 channels

Then: shape = (2, 3, 3)

Conceptually:

```  text

ROW 1

Pixel 1        Pixel 2        Pixel 3
[R,G,B]        [R,G,B]        [R,G,B]


ROW 2

Pixel 1        Pixel 2        Pixel 3
[R,G,B]        [R,G,B]        [R,G,B]

```
For example:

[
  [
    [255, 0, 0],
    [0, 255, 0],
    [0, 0, 255]
  ],

  [
    [255, 255, 255],
    [0, 0, 0],
    [128, 128, 128]
  ]
]

Shape:

2 × 3 × 3

height × width × channels

##### 8. RGB channels are separate numerical planes

Although we often write: [120, 85, 60]

for one pixel, another way to think about an image is as three matrices stacked together.

Suppose we have: 2 × 3 RGB image

``` text

Red channel:

255    0    0
255    0   128

Green channel:

0    255    0
255    0   128

Blue channel:

0      0   255
255    0   128

Together:

Red matrix
    +
Green matrix
    +
Blue matrix
    ↓
RGB image

```

This idea of channels becomes very important when we start CNNs.

##### 9. Grayscale representation

Now let's convert the image to grayscale:

gray_image = image.convert("L")

print(gray_image.mode)

Output: L

L represents a grayscale intensity mode in Pillow.

Now convert it to NumPy:

gray_array = np.array(gray_image)

print(gray_array.shape)

Instead of: (480, 640, 3)

we get: (480, 640)

Why?

Because we no longer have:

- Red
- Green
- Blue

We have only: Intensity

So:

- RGB image : Height × Width × 3
- Grayscale image: Height × Width

##### 10. Inspecting grayscale pixels

Suppose:

print(gray_array[0:4, 0:4])

returns:

``` text

[[ 25  30  32  35]
 [ 40  48  55  60]
 [ 65  72  80  90]
 [100 120 150 190]]

```
Remember:

- 0   → black
- 255 → white

and values between them represent different shades of gray.

So:

- 25  → very dark gray
- 100 → darker/mid gray
- 150 → lighter gray
- 230 → very light gray

##### 11. Why convert images to tensors?

NumPy arrays are useful for manipulating and inspecting images.

But our PyTorch neural network expects PyTorch tensors.

So eventually we need:

``` text

Image
 ↓
NumPy array
 ↓
Tensor
 ↓
CNN

```

A tensor is essentially a multidimensional numerical structure that PyTorch can operate on efficiently and use with automatic differentiation.

You've already used tensors in your neural-network projects.

For example:

Support Ticket NLP

Embedding tensor
[16, 384]

Now we'll use tensors containing image information.

##### 12. NumPy vs PyTorch shape

Here's an important difference.

Our NumPy RGB image might have: [height, width, channels] --> [480, 640, 3]

But PyTorch CNNs normally expect: [channels, height, width] --> [3, 480, 640]

So:
``` text

NumPy

[480, 640, 3]
      ↓
PyTorch
[3, 480, 640]

```

This rearrangement does not change the image.

We're simply changing how the dimensions are organized.

##### 13. Converting manually to a PyTorch tensor

For learning purposes, we can do this manually:

import torch

image_tensor = torch.from_numpy(image_array)

print(image_tensor.shape)

Initially:

torch.Size([480, 640, 3])

That's still: H × W × C

We can rearrange it:

image_tensor = image_tensor.permute(2, 0, 1)

print(image_tensor.shape)

Now: torch.Size([3, 480, 640])

permute(2, 0, 1) means:

``` text

Original dimensions

0 = Height
1 = Width
2 = Channels

        ↓

New order

2 = Channels
0 = Height
1 = Width

Therefore:

H, W, C

↓

C, H, W

```

##### 14. Adding the batch dimension

Here's another connection to your previous neural-network work.

A single image: [3, 224, 224]

But neural networks usually train with multiple images at once.

Suppose:

batch size = 32

Then the model receives: [32, 3, 224, 224]

Meaning:

- 32  = number of images
- 3   = channels
- 224 = height
- 224 = width

PyTorch CNN input convention is therefore commonly: [N, C, H, W]

where:

- N = batch size
- C = channels
- H = height
- W = width

This is worth remembering.

##### 15. Connection to your NLP tensors

This should make the transition from NLP easier.

In your text neural network, you had:

Batch of sentence embeddings

[16, 384]

- 16  = tickets
- 384 = embedding features

Now:

Batch of images

[16, 3, 224, 224]

- 16  = images
- 3   = RGB channels
- 224 = height
- 224 = width

So both are tensors.

The shape and meaning of the dimensions are different.

##### 16. Pixel values and normalization

Initially, our image values may be: 0 → 255

For neural-network training, we commonly transform them.

One simple transformation is: pixel / 255

For example:

``` text

Original      Normalized

0       →     0.000
64      →     0.251
128     →     0.502
192     →     0.753
255     →     1.000

```

So:

[0, 255]

↓

[0.0, 1.0]

Why?

Neural networks generally train more effectively when numerical inputs are kept at appropriate scales.

We'll cover image normalization properly in: 03_image_preprocessing

because pretrained models can require specific normalization rather than simply dividing by 255.

##### 17. 🔍 XAI connection — pixels matter

Here's an early XAI connection.

Our model ultimately bases its decision on patterns derived from these numerical image values.

Later, if the model predicts: Crack = 92%

we can investigate: Which spatial regions contributed strongly to that prediction?

That's where Grad-CAM eventually enters.

``` text

Pixels
   ↓
CNN features
   ↓
Prediction
   ↓
🔍 XAI
   ↓
Which regions/features influenced it?

```

##### 18. ⚖️ RAI connection — image quality matters

Suppose our training images are:

- Bright
- High resolution
- Centered
- Clean background

but production images are:

- Dark
- Blurry
- Different camera
- Different angle
- Dirty background

The model could perform much worse in production.

That's an early ⚖️ **Responsible AI — Reliability** concern.

We need to ask:

Does the training data reasonably represent the conditions where the model will actually be used?

We'll investigate these issues much more deeply in our RAI notebooks.

##### 19. One important concept before moving forward


``` text

At this point, I want you to visualize an RGB image like this:

             IMAGE

       Height × Width
             ↓
       ┌─────────────┐
Red    │             │
       └─────────────┘

       ┌─────────────┐
Green  │             │
       └─────────────┘

       ┌─────────────┐
Blue   │             │
       └─────────────┘

             ↓

PyTorch tensor

[3, Height, Width]

And multiple images become:

Batch
  ↓

[Batch, Channels, Height, Width]

[32, 3, 224, 224]

```

That tensor will eventually become the input to our CNN.

##### 20. Key Learnings

1. A digital image consists of pixels arranged spatially in a grid.

2. Image dimensions are described using width and height.

3. RGB images contain three color channels:
   Red, Green and Blue.

4. Each RGB pixel contains three numerical intensity values.

5. A grayscale image normally contains one intensity value per pixel.

6. NumPy commonly represents RGB images as:
   [height, width, channels].

7. PyTorch CNNs commonly represent an individual image as:
   [channels, height, width].

8. A batch of images is commonly represented as:
   [batch_size, channels, height, width].

9. For example:
   [32, 3, 224, 224]
   represents 32 RGB images of size 224 × 224.

10. Image pixel values are numerical model inputs and are commonly
    transformed/normalized before neural-network training.

11. 🔍 XAI investigates which image regions/features contributed to
    model predictions.

12. ⚖️ RAI includes checking whether training images adequately
    represent real production conditions so that model performance
    remains reliable.